In [ ]:
pip install psycopg2-binary sqlalchemy

In [ ]:
pip install python-dotenv

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine.url import URL
from urllib.parse import quote_plus

from dotenv import load_dotenv
import os



In [ ]:
df = pd.read_csv('customer_shopping_behavior.csv')

df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include = 'all')

In [ ]:
df.isnull().sum()

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns = {'purchase_amount_(usd)' : 'purchase_amount'})

In [ ]:
df.columns

In [ ]:
# create a column age_group

labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

In [ ]:
df[['age','age_group']].head(10)

In [ ]:
#create column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Annually': 365,
    'Bi-Weekly': 14,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

In [ ]:
df[['discount_applied', 'promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df = df.drop('promo_code_used', axis=1)

In [ ]:
df.columns

In [ ]:
#step 1" Connect to PostgreSQL
load_dotenv()

username = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
database = os.getenv("POSTGRES_DB")

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database
)

engine = create_engine(db_url)

# Step 2: Load dataframe into PostgreSQL
table_name = "customer_shopping_behavior"

df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"DataFrame loaded into PostgreSQL table '{table_name}' successfully.")